In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    "final_ppo_model/ppo_best.pt",
    map_location=device,
    weights_only=False
)


In [3]:
import torch
import torch.nn as nn

hidden_dim = 256          # MUST match training
action_dim = 3            # UREA, DAP, MOP
state_dim = len(checkpoint["feature_cols"])  # safe way

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.ones(action_dim) * -0.5)

    def forward(self, x):
        h = self.net(x)
        mu = self.mu(h)
        std = torch.exp(self.log_std)
        return mu, std


In [4]:
import torch
import numpy as np


feature_cols = checkpoint["feature_cols"]
fert_cols    = checkpoint["fert_cols"]
scaler_X     = checkpoint["scaler_X"]

actor = Actor().to(device)
actor.load_state_dict(checkpoint["actor_state_dict"])
actor.eval()


Actor(
  (net): Sequential(
    (0): Linear(in_features=9, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=256, out_features=3, bias=True)
)

In [ ]:
new_sample = {
    'ph': 6.4,
    'organic_matter': 2.1,
    'total_nitrogen': 0.18,
    'potassium': 210,
    'p2o5': 35,
    'zinc': 1.2,
    'sand': 42,
    'clay': 28,
    'slit': 30,
    'boron': 0.6,
    'lat': 27.7,
    'lon': 85.3,
    'parentsoil': 3,
    'crop': 1,
    'variety': 2,
    
}
# Convert to numpy array in training feature order
X_new = np.array([[new_sample[col] for col in feature_cols]])

# Apply scaler
X_new = scaler_X.transform(X_new)

# Convert to tensor
X_new = torch.tensor(X_new, dtype=torch.float32, device=device)

KeyError: 'UREA2'